***LAB 4***

Vacuum robot using reinforcement learning.
Same grid as Lab 3 (trap and recharge station as terminal cells), but this time the robot
**does not know T or R**. It starts in the top-left corner and can only learn by trying actions
and observing what happens. This notebook builds that environment and collects experience;
Q-learning is implemented in a later lab.

In [1]:
import os
import random
import pandas as pd
from collections import Counter
from IPython.display import display

MOVES = {
    "up": (-1, 0),
    "down": (+1, 0),
    "left": (0, -1),
    "right": (0, +1),
}

SLIPS = {
    "up": ("left", "right"),
    "down": ("right", "left"),
    "left": ("down", "up"),
    "right": ("up", "down"),
}

X = None  # obstacle
GRID = [
    [0, 0,  0, 0],
    [0, X,  0, 0],
    [0, 0,  0, 0],
    [0, -1, 0, 10],
]

START = (0, 0)          # Lab 4 starts top-left (Lab 3 started at (0, 3))
CHARGER = (3, 3)
TRAP = (3, 1)
TERMINALS = {CHARGER, TRAP}

GAMMA = 0.9
LIVING_REWARD = -0.04   # best setting from Lab 3
P_INTENDED, P_SLIP = 0.8, 0.1

In [2]:
def show_grid(grid, title):
    # S = start, X = obstacle
    print(title)
    for r, row in enumerate(grid):
        cells = []
        for c, value in enumerate(row):
            mark = "S" if (r, c) == START else " "
            text = "X" if value is None else value
            cells.append(f"{text:>4}{mark}")
        print("  " + "".join(cells))
    print()

show_grid(GRID, "LAB 4 GRID")

LAB 4 GRID
     0S   0    0    0 
     0    X    0    0 
     0    0    0    0 
     0   -1    0   10 



## The environment

In Lab 3, `VacuumMDP` exposed `T(s, a)` and `R(s)` and value iteration used them directly.
Here the model is hidden inside `VacuumEnv` (underscore attributes). The robot can only call
`reset()` and `step(action)`, which returns a sample `(s', r, done)`. That is the reinforcement
learning setting: unknown T and R, so the agent must try actions and learn from outcomes.

In [3]:
class VacuumEnv:
    def __init__(self, grid, living_reward=LIVING_REWARD, seed=None):
        # Hidden model: the agent never reads these directly
        self._grid = grid
        self._living_reward = living_reward
        self._rng = random.Random(seed)
        # Public: what any robot knows about its world
        self.states = [(r, c) for r in range(len(grid))
                       for c in range(len(grid[0])) if grid[r][c] is not None]
        self.terminals = TERMINALS
        self.state = None

    def actions(self, s):
        return [] if s in self.terminals else list(MOVES)

    def reset(self, start=START):
        self.state = start
        return self.state

    def _go(self, s, direction):
        # Move one cell; walls and the obstacle keep the robot in place
        r, c = s
        dr, dc = MOVES[direction]
        nxt = (r + dr, c + dc)
        return nxt if nxt in self.states else s

    def step(self, action):
        # Returns (next_state, reward, done, info). info is for our logs only.
        if self.state in self.terminals:
            raise RuntimeError("Episode is over, call reset()")
        left, right = SLIPS[action]
        actual = self._rng.choices([action, left, right],
                                   weights=[P_INTENDED, P_SLIP, P_SLIP])[0]
        nxt = self._go(self.state, actual)
        reward = self._living_reward
        if nxt in self.terminals:
            r, c = nxt
            reward += self._grid[r][c]
        done = nxt in self.terminals
        self.state = nxt
        return nxt, reward, done, dict(actual=actual)

## Exploring the world

With no T or R the robot has nothing to plan with, so it explores with random actions
(ε-greedy with ε = 1). Every step produces one sample transition `(s, a, r, s')`.
The `Actual` column comes from `info` and is only for us; the robot only sees where it ended up.

In [4]:
def random_policy(seed=None):
    rng = random.Random(seed)
    return lambda env, s: rng.choice(env.actions(s))

def run_episode(env, policy, start=START, max_steps=50, verbose=True):
    state, total, log, path = env.reset(start), 0.0, [], [start]
    for step in range(1, max_steps + 1):
        action = policy(env, state)
        nxt, reward, done, info = env.step(action)
        total += reward
        log.append(dict(Step=step, From=state, Action=action, Actual=info["actual"],
                        To=nxt, Note=" ".join(n for n, hit in (("slipped", info["actual"] != action),
                                                              ("stopped", nxt == state)) if hit),
                        Reward=round(reward, 2), Total=round(total, 2)))
        state = nxt
        path.append(state)
        if done:
            break

    if state == CHARGER:  outcome = "RECHARGED at the charger"
    elif state == TRAP:   outcome = "STUCK in the trap"
    else:                 outcome = f"never went home (stopped after {max_steps} steps)"

    if verbose:
        print("PATH LOG")
        display(pd.DataFrame(log).style.hide(axis="index").format(precision=2))
        print(f"\nPath: {' -> '.join(map(str, path))}")
        print(f"Outcome: {outcome}   Steps: {len(log)}   Total reward: {total:.2f}")
    return dict(path=path, log=log, outcome=outcome, total=total)

## Episode 1

First look at a random episode from the start cell.

In [5]:
ep1 = run_episode(VacuumEnv(GRID, seed=17), random_policy(seed=17))

PATH LOG


Step,From,Action,Actual,To,Note,Reward,Total
1,"(0, 0)",right,right,"(0, 1)",,-0.04,-0.04
2,"(0, 1)",left,down,"(0, 1)",slipped stopped,-0.04,-0.08
3,"(0, 1)",left,up,"(0, 1)",slipped stopped,-0.04,-0.12
4,"(0, 1)",left,left,"(0, 0)",,-0.04,-0.16
5,"(0, 0)",down,down,"(1, 0)",,-0.04,-0.20
6,"(1, 0)",left,left,"(1, 0)",stopped,-0.04,-0.24
7,"(1, 0)",up,up,"(0, 0)",,-0.04,-0.28
8,"(0, 0)",up,up,"(0, 0)",stopped,-0.04,-0.32
9,"(0, 0)",down,down,"(1, 0)",,-0.04,-0.36
10,"(1, 0)",right,right,"(1, 0)",stopped,-0.04,-0.40



Path: (0, 0) -> (0, 1) -> (0, 1) -> (0, 1) -> (0, 0) -> (1, 0) -> (1, 0) -> (0, 0) -> (0, 0) -> (1, 0) -> (1, 0) -> (1, 0) -> (1, 0) -> (1, 0) -> (1, 0) -> (1, 0) -> (1, 0) -> (2, 0) -> (2, 0) -> (3, 0) -> (3, 1)
Outcome: STUCK in the trap   Steps: 20   Total reward: -1.80


## Episode 2

A second random episode with a different seed.

In [6]:
ep2 = run_episode(VacuumEnv(GRID, seed=4), random_policy(seed=4))

PATH LOG


Step,From,Action,Actual,To,Note,Reward,Total
1,"(0, 0)",down,down,"(1, 0)",,-0.04,-0.04
2,"(1, 0)",left,left,"(1, 0)",stopped,-0.04,-0.08
3,"(1, 0)",up,up,"(0, 0)",,-0.04,-0.12
4,"(0, 0)",right,right,"(0, 1)",,-0.04,-0.16
5,"(0, 1)",right,right,"(0, 2)",,-0.04,-0.20
6,"(0, 2)",down,down,"(1, 2)",,-0.04,-0.24
7,"(1, 2)",up,right,"(1, 3)",slipped,-0.04,-0.28
8,"(1, 3)",up,left,"(1, 2)",slipped,-0.04,-0.32
9,"(1, 2)",up,up,"(0, 2)",,-0.04,-0.36
10,"(0, 2)",right,right,"(0, 3)",,-0.04,-0.40



Path: (0, 0) -> (1, 0) -> (1, 0) -> (0, 0) -> (0, 1) -> (0, 2) -> (1, 2) -> (1, 3) -> (1, 2) -> (0, 2) -> (0, 3) -> (0, 2) -> (0, 2) -> (1, 2) -> (1, 2) -> (1, 2) -> (1, 2) -> (1, 2) -> (2, 2) -> (2, 3) -> (1, 3) -> (1, 2) -> (1, 2) -> (2, 2) -> (2, 3) -> (3, 3)
Outcome: RECHARGED at the charger   Steps: 25   Total reward: 9.00


## Episode 3

A third random episode with a different seed.

In [7]:
ep3 = run_episode(VacuumEnv(GRID, seed=23), random_policy(seed=23))

PATH LOG


Step,From,Action,Actual,To,Note,Reward,Total
1,"(0, 0)",left,up,"(0, 0)",slipped stopped,-0.04,-0.04
2,"(0, 0)",up,right,"(0, 1)",slipped,-0.04,-0.08
3,"(0, 1)",up,left,"(0, 0)",slipped,-0.04,-0.12
4,"(0, 0)",left,left,"(0, 0)",stopped,-0.04,-0.16
5,"(0, 0)",right,right,"(0, 1)",,-0.04,-0.20
6,"(0, 1)",right,right,"(0, 2)",,-0.04,-0.24
7,"(0, 2)",left,left,"(0, 1)",,-0.04,-0.28
8,"(0, 1)",down,down,"(0, 1)",stopped,-0.04,-0.32
9,"(0, 1)",down,down,"(0, 1)",stopped,-0.04,-0.36
10,"(0, 1)",left,left,"(0, 0)",,-0.04,-0.40



Path: (0, 0) -> (0, 0) -> (0, 1) -> (0, 0) -> (0, 0) -> (0, 1) -> (0, 2) -> (0, 1) -> (0, 1) -> (0, 1) -> (0, 0) -> (0, 1) -> (0, 1) -> (0, 1) -> (0, 2) -> (0, 2) -> (0, 2) -> (0, 2) -> (0, 3) -> (0, 3) -> (0, 3) -> (0, 3) -> (0, 2) -> (0, 2) -> (1, 2) -> (1, 3) -> (1, 2) -> (2, 2) -> (3, 2) -> (3, 1)
Outcome: STUCK in the trap   Steps: 29   Total reward: -2.16


## Collecting experience

One episode says little. Running many episodes and keeping every `(s, a, r, s')` sample is the
raw material Q-learning will use later.

In [8]:
def collect_samples(n_episodes=500, max_steps=50, seed=0):
    env = VacuumEnv(GRID, seed=seed)
    policy = random_policy(seed=seed)
    samples, outcomes, returns = [], Counter(), []
    for episode in range(1, n_episodes + 1):
        run = run_episode(env, policy, max_steps=max_steps, verbose=False)
        outcomes[run["outcome"].split(" (")[0]] += 1
        returns.append(run["total"])
        samples += [dict(Episode=episode, s=row["From"], a=row["Action"],
                         r=row["Reward"], s2=row["To"]) for row in run["log"]]
    return samples, outcomes, returns

samples, outcomes, returns = collect_samples()
print(f"Episodes: {sum(outcomes.values())}   Samples (s, a, r, s'): {len(samples)}")
print(f"Average return of a random robot: {sum(returns) / len(returns):.2f}\n")
display(pd.DataFrame(outcomes.items(), columns=["Outcome", "Episodes"])
        .style.hide(axis="index"))

Episodes: 500   Samples (s, a, r, s'): 14406
Average return of a random robot: 1.04



Outcome,Episodes
RECHARGED at the charger,135
STUCK in the trap,256
never went home,109


## What the samples tell us

The slides' big idea: compute averages over T using sample outcomes. Counting where
`right` from (2, 2) actually landed should approach the hidden 80% / 10% / 10% split,
even though the robot was never told those numbers.

In [9]:
def estimate_T(samples, s, a):
    landed = Counter(x["s2"] for x in samples if x["s"] == s and x["a"] == a)
    n = sum(landed.values())
    return pd.DataFrame([dict(Next=s2, Count=k, Estimated_P=round(k / n, 3))
                         for s2, k in landed.most_common()])

def observed_rewards(samples):
    # Average reward seen when landing in each cell
    by_cell = {}
    for x in samples:
        by_cell.setdefault(x["s2"], []).append(x["r"])
    return pd.DataFrame(sorted((dict(Landed_in=c, Visits=len(v), Avg_reward=round(sum(v) / len(v), 2))
                                for c, v in by_cell.items()), key=lambda d: d["Landed_in"]))

print("Estimated T((2, 2), right, s')")
display(estimate_T(samples, (2, 2), "right").style.hide(axis="index").format(precision=3))
print("\nRewards the robot observed")
display(observed_rewards(samples).style.hide(axis="index").format(precision=2))

Estimated T((2, 2), right, s')


Next,Count,Estimated_P
"(2, 3)",78,0.821
"(1, 2)",9,0.095
"(3, 2)",8,0.084



Rewards the robot observed


Landed_in,Visits,Avg_reward
"(0, 0)",2628,-0.04
"(0, 1)",2319,-0.04
"(0, 2)",1384,-0.04
"(0, 3)",1009,-0.04
"(1, 0)",2142,-0.04
"(1, 2)",864,-0.04
"(1, 3)",743,-0.04
"(2, 0)",947,-0.04
"(2, 1)",497,-0.04
"(2, 2)",470,-0.04


## Saving the logs

In [10]:
def save_logs(run, name, folder="logs"):
    os.makedirs(folder, exist_ok=True)
    df = pd.DataFrame(run["log"])
    df["From"] = df["From"].astype(str)
    df["To"] = df["To"].astype(str)
    df["Outcome"] = ""
    df.loc[df.index[-1], "Outcome"] = run["outcome"]
    path = f"{folder}/lab4_log_{name}.csv"
    df.to_csv(path, index=False)
    return path

for i, run in enumerate((ep1, ep2, ep3), start=1):
    print("saved", save_logs(run, f"episode{i}"))

df = pd.DataFrame(samples)
df["s"], df["s2"] = df["s"].astype(str), df["s2"].astype(str)
df.to_csv("logs/lab4_samples.csv", index=False)
print("saved logs/lab4_samples.csv")

saved logs/lab4_log_episode1.csv
saved logs/lab4_log_episode2.csv
saved logs/lab4_log_episode3.csv


saved logs/lab4_samples.csv


## Results and discussion

- **Random exploration is a bad policy.** Out of 500 episodes the random robot recharged 135 times (27%), fell in the trap 256 times (51%) and wandered for 50 steps without finishing 109 times (22%). Its average return was only 1.04. The trap is closer to the left column where the robot starts, so random moves find it first.
- **The robot can rebuild T from samples.** For `right` from (2, 2) it landed on (2, 3) 82% of the time and slipped to (1, 2) or (3, 2) about 9% each, close to the hidden 80/10/10 split it was never told.
- **The robot can rebuild R from samples.** Every ordinary cell averaged -0.04, the trap -1.04 and the charger 9.96 (living cost plus the terminal value).
- **Why this matters.** In Lab 3 value iteration had T and R and planned the best path immediately (U(0, 0) = 4.35). Here the robot only has experience. The samples saved in `logs/lab4_samples.csv` are the input Q-learning will use to learn a policy without ever building T or R.

## Next: Q-learning

Each saved row `(s, a, r, s')` is exactly what the Q-learning update consumes:

Q(s, a) ← (1 − α) Q(s, a) + α [ r + γ max<sub>a'</sub> Q(s', a') ]

The next lab adds a `QLearningAgent` that replaces `random_policy` with ε-greedy action selection
and applies this update after every `env.step()`.